In [ ]:
from datasets import load_dataset

dataset = load_dataset("jlh-ibm/earnings_call", "transcripts")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0].keys())
print(train_data[0]["company"])
print(train_data[0]["date"])
print(train_data[0]["transcript"][:1000])

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38 [00:00<?, ? examples/s]

dict_keys(['company', 'date', 'transcript'])
AMD
2016-07-21


Thomson Reuters StreetEvents Event Transcript
E D I T E D   V E R S I O N

Q2 2016 Advanced Micro Devices Inc Earnings Call
JULY 21, 2016 / 9:00PM GMT

Corporate Participants

 * Lisa Su
   Advanced Micro Devices, Inc. - President and CEO
 * Devinder Kumar
   Advanced Micro Devices, Inc. - SVP, CFO and Treasurer
 * Ruth Cotter
   Advanced Micro Devices, Inc. - Chief Human Resources Officer and SVP of Corporate Communications and IR

Conference Call Participiants

 * Matt Ramsay
   Canaccord Genuity - Analyst
 * Jaguar Bajwa
   Arete Research - Analyst
 * Gabriel Ho
   BMO Capital Markets - Analyst
 * Mark Lipacis
   Jefferies LLC - Analyst
 * Stacy Rasgon
   Bernste


In [2]:
sample = dataset["train"][0]

print(sample["company"])
print(sample["date"])
print(sample["transcript"][:3000])

AMD
2016-07-21


Thomson Reuters StreetEvents Event Transcript
E D I T E D   V E R S I O N

Q2 2016 Advanced Micro Devices Inc Earnings Call
JULY 21, 2016 / 9:00PM GMT

Corporate Participants

 * Lisa Su
   Advanced Micro Devices, Inc. - President and CEO
 * Devinder Kumar
   Advanced Micro Devices, Inc. - SVP, CFO and Treasurer
 * Ruth Cotter
   Advanced Micro Devices, Inc. - Chief Human Resources Officer and SVP of Corporate Communications and IR

Conference Call Participiants

 * Matt Ramsay
   Canaccord Genuity - Analyst
 * Jaguar Bajwa
   Arete Research - Analyst
 * Gabriel Ho
   BMO Capital Markets - Analyst
 * Mark Lipacis
   Jefferies LLC - Analyst
 * Stacy Rasgon
   Bernstein Research - Analyst
 * David Wong
   Wells Fargo Securities, LLC - Analyst
 * Marco Chan
   Citigroup - Analyst
 * Ian Ing
   MKM Partners - Analyst
 * Sidney Ho
   Deutsche Bank - Analyst
 * Joe Moore
   Morgan Stanley - Analyst
 * John Pitzer
   Credit Suisse Group - Analyst

Presentation
---------------

In [4]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

transcript = sample["transcript"]

# 1. Presentation부터 시작
start = transcript.find("Presentation")
if start != -1:
    transcript = transcript[start:]

# 2. Definitions/Disclaimer 이후 제거
end_patterns = ["Definitions", "Disclaimer", "Copyright"]
for pattern in end_patterns:
    idx = transcript.find(pattern)
    if idx != -1:
        transcript = transcript[:idx]

# 3. 너무 많은 구분선 정리
transcript = re.sub(r"={5,}", " ", transcript)
transcript = re.sub(r"-{5,}", " ", transcript)

doc = nlp(transcript)
sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 20]

print(len(sentences))
print(sentences[:20])

459
["Presentation\n \nOperator    [1]\n \nGreetings, and welcome to the Advanced Micro Devices' second-quarter 2016 earnings conference call.", 'At this time, all participants are in a listen-only mode.', 'A question-and-answer session will follow the formal presentation.', '(Operator Instructions)', 'As a reminder, this conference is being recorded.', 'I would now like to turn the conference over to your host, Ruth Cotter, Chief Human Resources Officer, and Senior Vice President of Corporate Communications and Investor Relations.', "Ruth Cotter,  Advanced Micro Devices, Inc. - Chief Human Resources Officer and SVP of Corporate Communications and IR    [2]\n \nThank you and welcome to AMD's second-quarter conference call.", 'By now you should have had the opportunity to review a copy of our earnings release and the CFO commentary and slides.', "If you've not reviewed these documents, they can be found on AMD's website at IOR.AMD.com.", "Participants on today's conference call are Lisa

In [5]:
import re

cleaned_sentences = []

noise_patterns = [
    r"Operator Instructions",
    r"conference is being recorded",
    r"listen-only mode",
]

for sent in sentences:

    # speaker metadata 제거
    sent = re.sub(r".*?\[\d+\]\s*", "", sent)

    # 공백 정리
    sent = sent.strip()

    # 너무 짧은 문장 제거
    if len(sent) < 30:
        continue

    # noise 제거
    if any(pattern.lower() in sent.lower() for pattern in noise_patterns):
        continue

    cleaned_sentences.append(sent)

print(len(cleaned_sentences))
print(cleaned_sentences[:20])

421
["Presentation\n \nGreetings, and welcome to the Advanced Micro Devices' second-quarter 2016 earnings conference call.", 'A question-and-answer session will follow the formal presentation.', 'I would now like to turn the conference over to your host, Ruth Cotter, Chief Human Resources Officer, and Senior Vice President of Corporate Communications and Investor Relations.', "Thank you and welcome to AMD's second-quarter conference call.", 'By now you should have had the opportunity to review a copy of our earnings release and the CFO commentary and slides.', "If you've not reviewed these documents, they can be found on AMD's website at IOR.AMD.com.", "Participants on today's conference call are Lisa Su, our President and Chief Executive Officer; and Devinder Kumar, our Senior Vice President, Chief Financial Officer and Treasurer.", 'This is a live call and will be replayed via webcast on AMD.com.', "I'd like to take the opportunity to highlight a few dates for you.", 'Lisa Su will pr

In [8]:
print("""
===== RAW TRANSCRIPT =====

Operator [1]

Greetings, and welcome to the Advanced Micro Devices'
second-quarter 2016 earnings conference call.

At this time, all participants are in a listen-only mode.

A question-and-answer session will follow the formal presentation.

(Operator Instructions)

As a reminder, this conference is being recorded.

Ruth Cotter, Advanced Micro Devices, Inc. [2]

Thank you and welcome to AMD's second-quarter conference call.

Revenue increased 12% YoY during the quarter.

We expect continued growth in the data center market next quarter.

Currency headwinds affected international sales.
""")


===== RAW TRANSCRIPT =====

Operator [1]

Greetings, and welcome to the Advanced Micro Devices'
second-quarter 2016 earnings conference call.

At this time, all participants are in a listen-only mode.

A question-and-answer session will follow the formal presentation.

(Operator Instructions)

As a reminder, this conference is being recorded.

Ruth Cotter, Advanced Micro Devices, Inc. [2]

Thank you and welcome to AMD's second-quarter conference call.

Revenue increased 12% YoY during the quarter.

We expect continued growth in the data center market next quarter.

Currency headwinds affected international sales.



In [9]:
print("""
===== CLEAN SENTENCES =====

[
    "Revenue increased 12% YoY during the quarter.",

    "We expect continued growth in the data center market next quarter.",

    "Currency headwinds affected international sales."
]
""")


===== CLEAN SENTENCES =====

[
    "Revenue increased 12% YoY during the quarter.",

    "We expect continued growth in the data center market next quarter.",

    "Currency headwinds affected international sales."
]



In [10]:
print("""
===== INPUT SENTENCE =====

We expect continued growth
in the data center market
next quarter.
""")


===== INPUT SENTENCE =====

We expect continued growth
in the data center market
next quarter.



In [ ]:
print("""
===== PROMPT =====

You are a financial information extraction model.

Extract the financial semantic label
from the given sentence.

Available Labels:
- REVENUE
- GUIDANCE
- RISK
- GROWTH_SIGNAL


Example 1:
Sentence:
"Revenue increased 12% YoY."

Output:
{
    "label": "REVENUE"
}


Example 2:
Sentence:
"Currency headwinds affected results."

Output:
{
    "label": "RISK"
}


Sentence:
"We expect continued growth
in the data center market next quarter."

Return JSON format only.
""")

In [12]:
print("""
===== EXAMPLE 1 =====

Input Sentence:
"Revenue increased 12% YoY."

Extracted Label:
REVENUE

Structured Output:

{
    "label": "REVENUE",

    "text":
    "Revenue increased 12% YoY"
}
""")


===== EXAMPLE 1 =====

Input Sentence:
"Revenue increased 12% YoY."

Extracted Label:
REVENUE

Structured Output:

{
    "label": "REVENUE",

    "text":
    "Revenue increased 12% YoY"
}

